# ML-03 — Frame Your Lane as an ML Task

**Lane: Lane 2 — Refresh / Content Opportunity Scoring.**

This frame was built with the `framing-ml-problems` and `flyrank/flyrank-data` skills.
Every number below is computed live from `data/raw/content_refresh_anonymized.csv` —
nothing hardcoded.

## 1. My lane as an ML task (type)

**Ranking / scoring.** The question "which page should an editor refresh first?" needs an
*order* of preference, not a category. It is not classification (no single yes/no category)
and not regression (no continuous number) — the output is a **prioritized queue** of pages,
ordered by how urgently each needs a refresh.

The queue is the product because the review workload dwarfs human capacity.

In [1]:
import os
import pandas as pd

ROOT = os.getcwd()
CSV = os.path.join("data", "raw", "content_refresh_anonymized.csv")
while not os.path.exists(os.path.join(ROOT, CSV)):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        raise FileNotFoundError(
            f"{CSV} not found from {os.getcwd()} — open this notebook from inside the Assignment-1 repo"
        )
    ROOT = parent

df = pd.read_csv(os.path.join(ROOT, "data", "raw", "content_refresh_anonymized.csv"))

visible = (df["impressions_90d"] >= 500).sum()
capacity_week = 20
print(f"visible pages (impressions_90d >= 500): {visible:,}")
print(f"review capacity per week: {capacity_week}")
print(f"-> {visible:,} candidates for {capacity_week} weekly slots: an ORDER is the product, not a flag.")

visible pages (impressions_90d >= 500): 16,726
review capacity per week: 20
-> 16,726 candidates for 20 weekly slots: an ORDER is the product, not a flag.


## 2. Target or proxy

**Target:** a page's *future decline risk* — whether impressions 30 days from the decision
point fall by more than 20%. The underlying outcome (impressions 30 days out) is **observed**
in the warehouse data; the "more than 20%" is **our cutoff** on top. The rule's only job is to
convert a continuous observed measurement into a yes/no label — that cutoff is legitimate.

It is **not** the rule-defined `trend_direction` bucket, which is computed from `trend_pct` by
the prep pipeline. Predicting that would mean learning a rule, not the world. The check below
proves `trend_direction` is a derived rule (not an observed outcome) and shows how many rows it
swallows as the starter label.

In [2]:
buckets = df.groupby("trend_direction")["content_id"].count().sort_values(ascending=False)
print(buckets.to_string())

down = df["trend_direction"] == "down"
consistent = (df.loc[down, "trend_pct"] < -20).sum()
print(f"\nof {down.sum():,} 'down' rows, {consistent:,} have trend_pct < -20")
print("-> trend_direction is derived from trend_pct by a rule; it is a proxy, not an observed outcome.")

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152

of 16,262 'down' rows, 16,258 have trend_pct < -20
-> trend_direction is derived from trend_pct by a rule; it is a proxy, not an observed outcome.


## 3. Success metric

**Precision@20** — the fraction of the top 20 pages in the queue that actually declined in the
next 30 days. This matches the real decision: the editor has limited hours, so what matters is
whether the *top of the list* (where hours are spent) is right. Accuracy over all 200 pages
would be diluted by the ~180 pages nobody acts on.

A fair baseline to beat: a random order would achieve precision@20 equal to the base rate —
the fraction of the eligible population that declines. Our model must beat that.

In [3]:
eligible = df[df["impressions_90d"] >= 500]
base_rate = (eligible["trend_direction"] == "down").mean()
print(f"eligible visible pages: {len(eligible):,}")
print(f"declining rate in the eligible population: {base_rate:.3f}")
print(f"-> random top-20 precision would be ~{base_rate:.2f}; the ranked queue must beat that.")

eligible visible pages: 16,726
declining rate in the eligible population: 0.596
-> random top-20 precision would be ~0.60; the ranked queue must beat that.


## 4. The unit of analysis, as a real dataframe

One row = **one content item (page)** — the unit an editor decides about, and the unit the
queue ranks. Grain check: row count equals unique `content_id` count, so no page is duplicated
or aggregated.

In [4]:
print(f"rows: {len(df):,}   unique content_id: {df['content_id'].nunique():,}   unique client_id: {df['client_id'].nunique():,}")
grain_ok = len(df) == df["content_id"].nunique()
print(f"grain check (one row per content item): {grain_ok}")
df.head(5)

rows: 30,000   unique content_id: 30,000   unique client_id: 32
grain check (one row per content item): True


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 5. Why ML beats a fixed rule here

A hand-written rule uses hard cutoffs (e.g. `impressions_90d >= 500`) and needs a separate
if-statement for every combination of signals that matters. With ~15 interacting signals —
impressions, position, trend, freshness, engagement, content type — the combinations explode,
and a page just below a cutoff with a *severe* decline signal is invisible to the rule. A model
weights all signals together smoothly and learns interactions without us hand-writing them.

The check below shows the cliff: how many pages sit just below the 500-impression cutoff and
are declining — pages a pure rule would miss.

In [ ]:
just_below = df[(df["impressions_90d"] >= 400) & (df["impressions_90d"] < 500)]
just_below_down = (just_below["trend_direction"] == "down").sum()
print(f"pages with 400-499 impressions: {len(just_below):,}")
print(f"of those, declining: {just_below_down:,}")
print("-> a rule cut at 500 misses every one of these; a model weights the decline signal instead of a cliff.")

pages with 400-499 impressions: 908
of those, declining: 529
-> a rule cut at 500 misses every one of these; a model weights the decline signal instead of a cliff.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.